Este programa sirve para realizar la descomposición $LU$ de una matriz $A$ con el método de Dolittle. $L$ es una matriz triangular inferior y $U$ una triangular superior (una de ellas debe tener $1$'s en la diagonal, la otra no). El procedimiento exacto está explicado en el tema 3 de MEN.

In [11]:
import pandas as pd              # Para análisis de datos
import numpy as np               # Para cálculos numéricos
from scipy import linalg 

Hay que ir calculando $M^{(k)}$ para poder calcular $A^{(k+1)}$. Hay tantas $k$'s como columnas tenga $A$, y cada $M^{(k)}$ trabaja en una de las columnas solamente. Esta $M$ se calcula como:

$m_{j1}^{(1)} = a_{j1}^{(1)} / a_{11}^{(1)}$ ,

$m_{jk}^{(k)} = a_{jk}^{(k)} / a_{kk}^{(k)}$ .

Entonces podemos calcular:

Se cumple que $U = A^{(n)} = M^{(n-1)}M^{(n-2)}...M^{(2)}M^{(1)}A$, habiendo tantas $n$'s como columnas (o filas, ya que trabajamos con dim $N\times N$).

Se cumple que $L = [M^{(1)}]^{-1} [M^{(2)}]^{-1} ... [M^{(n-2)}]^{-1}[M^{(n-1)}]^{-1}$ .

In [14]:
def descompLU(A):
    # A = matriz a descomponer en LU

    dimfilas, dimcolum = A.shape

    if dimfilas == dimcolum:

        dim = dimfilas

        M = np.eye(dim) # Matriz identidad de dimension dim x dim
        L = np.eye(dim)

        for k in range (0,dim-1,1): # En la ultima columna no hacemos nada 
            for j in range (k+1,dim,1): # En la diagonal de M siempre debe haber un 1
                M[j][k] = - A[j][k] / A[k][k]
            
            A = np.dot(M,A)

            L = np.dot(L,np.linalg.inv(M))

            M = np.eye(dim) # Volvemos a M unidad


        return L,A # A^n = U, y L

    else:
        print('Error: A no es cuadrada.')
        return 

Vamos a probar si funciona. La matriz a descomponer es:

$A = \begin{pmatrix} 
1 & 2 & 4 & 1 \\ 
2 & 8 & 6 & 4 \\
3 & 10 & 8 & 8 \\
4 & 12 & 10 & 6
\end{pmatrix}$

In [17]:
A = np.array([[1,2,4,1],[2,8,6,4],[3,10,8,8],[4,12,10,6]],dtype=float)

L,U = descompLU(A)

print('L =',L)
print('U =',U)
print('A = LU =',np.dot(L,U))

L = [[1. 0. 0. 0.]
 [2. 1. 0. 0.]
 [3. 1. 1. 0.]
 [4. 1. 2. 1.]]
U = [[ 1.  2.  4.  1.]
 [ 0.  4. -2.  2.]
 [ 0.  0. -2.  3.]
 [ 0.  0.  0. -6.]]
A = LU = [[ 1.  2.  4.  1.]
 [ 2.  8.  6.  4.]
 [ 3. 10.  8.  8.]
 [ 4. 12. 10.  6.]]


Si ahora quisiéramos resolver el sistema de ecuaciones que genera $A$ podemos usar el metodo $LU$ para hacerlo más facilmente. Si tenemos 

$\begin{cases}
x + 2y + 4z + t = 21 \\
2x + 8y + 6z + 4t = 52\\
3x + 10y + 8z + 8t = 79\\
4x + 12y + 10z + 6t = 82
\end{cases}$ ,

lo que queremos resolver entonces es $A \vec{v} = \vec{b}$. La forma de hacer esto es hacer $A \vec{v} = LU \vec{v} = L \vec{y} = \vec{b}$ con $U \vec{v} = \vec{y}$. Entonces:

(1) $L \vec{y} = \vec{b}$

(2) $U \vec{v} = \vec{y}$

Entonces tenemos que resolver dos sistemas de ecuaciones triangulares, donde el despeje de las incógnitas es directo.

Haciendo un poco de mates obtengo que $y_k = b_k - \sum_{j=1}^{k-1}L_{kj}y_k$, con $k={1,2,...,dim(\vec{b})}$.

In [41]:
def vec_y(L,b):
    # L = matriz L de la descomposicion de A
    # b = vector de numeros del sistema

    dimfil,dimcol = L.shape

    if dimfil==dimcol:

        dim = dimfil

        y = np.zeros((dim,1))

        y[0][0] = b[0][0]

        for k in range (1,dim,1): # Empiezo en 1 pq el 0 ya lo tengo, y[0]

            sum = 0

            for j in range (0,k,1):
                sum = sum + L[k][j]*y[j][0]

            y[k][0] = b[k][0] - sum

        return y

    else:
        print('Error: L no es cuadrada.')
        return


Obtención de $\vec{y}$ a partir de $L$ y $\vec{b}$: 

In [42]:
b = np.array([[21],[52],[79],[82]])

y = vec_y(L,b)

print('y =',y)

y = [[ 21.]
 [ 10.]
 [  6.]
 [-24.]]


Una vez tenemos $\vec{y}$ ahora podemos calcular $\vec{v}$ haciendo un programa similar. Haciendo un poco de mates he obtenido que 

$v_k = (y_k - \sum_{j=k+1}^{dim(\vec{y})} U_{kj}v_k) / u_{kk}$, con $k=1,2,...,dim(\vec{y})$. 

In [39]:
def vec_v(U,y):
    # U = matriz U de la descomposicion LU
    # y = vector obtenido a partir de Ly=b

    dimfil,dimcol = U.shape

    if dimfil==dimcol:

        dim = dimfil

        v = np.zeros((dim,1))

        v[dim-1][0] = y[dim-1][0] / U[dim-1][dim-1]

        for k in range (dim-2,-1,-1): #Pongo que llegue hasta -1 para que llegue hasta v[0]

            sum = 0

            for j in range (k+1,dim,1):
                sum = sum + U[k][j]*v[j][0]

            v[k][0] = (y[k][0] - sum) / U[k][k]

        return v

    else:
        print('Error: U no es cuadrada.')
        return

In [43]:
sol_v = vec_v(U,y)

print('La solucion es v =',sol_v)

La solucion es v = [[1.]
 [2.]
 [3.]
 [4.]]
